# PAPI - Tiền xử lý dữ liệu

Notebook này thực hiện và trình bày trực tiếp toàn bộ logic gộp 14 tệp dữ liệu thô thành bộ dữ liệu
chuẩn dạng long, kèm diễn giải cho từng bước. Toàn bộ mã được viết ngay trong notebook để có thể đọc
và chạy độc lập.

Thư mục `src/` giữ một bản cài đặt tương đương của cùng logic, dùng cho ứng dụng dashboard và để đối
chiếu. Ô cuối cùng của notebook kiểm tra rằng kết quả của notebook trùng khớp với bản trong `src/`,
bảo đảm hai bản không lệch nhau.

Notebook này được chạy sau `data_understanding.ipynb`.

In [1]:
import os, re, unicodedata
import pandas as pd
ROOT = os.path.dirname(os.getcwd())
RAW = os.path.join(ROOT, 'data', 'raw'); PROC = os.path.join(ROOT, 'data', 'processed')
os.makedirs(PROC, exist_ok=True)
pd.set_option('display.max_columns', 30, 'display.width', 160)

## 1. Bảng tra cứu tỉnh và sáu vùng kinh tế - xã hội

In [2]:
REGIONS = {
    'Trung du và miền núi phía Bắc': ['Hà Giang','Cao Bằng','Bắc Kạn','Tuyên Quang','Lào Cai',
        'Yên Bái','Thái Nguyên','Lạng Sơn','Bắc Giang','Phú Thọ','Điện Biên','Lai Châu','Sơn La','Hòa Bình'],
    'Đồng bằng sông Hồng': ['Hà Nội','Vĩnh Phúc','Bắc Ninh','Quảng Ninh','Hải Dương','Hải Phòng',
        'Hưng Yên','Thái Bình','Hà Nam','Nam Định','Ninh Bình'],
    'Bắc Trung Bộ và Duyên hải miền Trung': ['Thanh Hóa','Nghệ An','Hà Tĩnh','Quảng Bình','Quảng Trị',
        'Thừa Thiên Huế','Đà Nẵng','Quảng Nam','Quảng Ngãi','Bình Định','Phú Yên','Khánh Hòa','Ninh Thuận','Bình Thuận'],
    'Tây Nguyên': ['Kon Tum','Gia Lai','Đắk Lắk','Đắk Nông','Lâm Đồng'],
    'Đông Nam Bộ': ['Bình Phước','Tây Ninh','Bình Dương','Đồng Nai','Bà Rịa - Vũng Tàu','Hồ Chí Minh'],
    'Đồng bằng sông Cửu Long': ['Long An','Tiền Giang','Bến Tre','Trà Vinh','Vĩnh Long','Đồng Tháp',
        'An Giang','Kiên Giang','Cần Thơ','Hậu Giang','Sóc Trăng','Bạc Liêu','Cà Mau'],
}

def norm(s):
    # Chuan hoa ten tinh: bo dau thanh, thuong hoa, doi d, bo tien to va dau cach
    if s is None: return ''
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = s.lower().strip().replace('đ', 'd')
    for pre in ('tp.', 'tp ', 'thanh pho ', 'tinh ', 'province of ', 'city of '):
        if s.startswith(pre): s = s[len(pre):]
    return re.sub(r'[\s\-\.]', '', s)

rows, pid, NORM2ID = [], 0, {}
for rid, (region, provs) in enumerate(REGIONS.items(), start=1):
    for name in provs:
        pid += 1; NORM2ID[norm(name)] = pid
        if name == 'Hồ Chí Minh': NORM2ID['hochiminhcity'] = pid
        if name == 'Thừa Thiên Huế': NORM2ID['hue'] = pid
        if name == 'Bà Rịa - Vũng Tàu': NORM2ID[norm('Ba Ria-Vung Tau')] = pid
        name_en = ''.join(c for c in unicodedata.normalize('NFD', name)
                          if unicodedata.category(c) != 'Mn').replace('đ', 'd')
        rows.append(dict(province_id=pid, province_vi=name, province_en=name_en, region=region, region_id=rid))
DIM_PROVINCE = pd.DataFrame(rows)
def to_pid(x): return NORM2ID.get(norm(x))
print('So tinh:', len(DIM_PROVINCE)); DIM_PROVINCE.head(3)

So tinh: 63


,province_id,province_vi,province_en,region,region_id
0,1,Hà Giang,Ha Giang,Trung du và miền núi phía Bắc,1
1,2,Cao Bằng,Cao Bang,Trung du và miền núi phía Bắc,1
2,3,Bắc Kạn,Bac Kan,Trung du và miền núi phía Bắc,1


**Nhận xét.** Bảng tra cứu gắn 63 tỉnh với sáu vùng. Hàm norm chuẩn hóa tên tỉnh để khớp giữa các tệp, xử lý khác biệt về dấu, ký tự Đ và các tiền tố như TP.

## 2. Bảng tra cứu tám trục

In [ ]:
DIM_INDICATOR = pd.DataFrame([
    dict(code='D1', name_vi='Tham gia của người dân ở cấp cơ sở',      name_en='Participation',            short='Tham gia',         color='#4C6A9C', sort=1, from_year=2011),
    dict(code='D2', name_vi='Công khai, minh bạch trong ra quyết định', name_en='Transparency',             short='Minh bạch',        color='#E0A23B', sort=2, from_year=2011),
    dict(code='D3', name_vi='Trách nhiệm giải trình với người dân',     name_en='Vertical Accountability',  short='Giải trình',       color='#578145', sort=3, from_year=2011),
    dict(code='D4', name_vi='Kiểm soát tham nhũng trong khu vực công',  name_en='Control of Corruption',    short='Chống tham nhũng', color='#B13507', sort=4, from_year=2011),
    dict(code='D5', name_vi='Thủ tục hành chính công',                  name_en='Public Admin. Procedures', short='Thủ tục HC',       color='#6D4C9C', sort=5, from_year=2011),
    dict(code='D6', name_vi='Cung ứng dịch vụ công',                    name_en='Public Service Delivery',  short='Dịch vụ công',     color='#2C8C99', sort=6, from_year=2011),
    dict(code='D7', name_vi='Quản trị môi trường',                      name_en='Environmental Governance', short='Môi trường',       color='#8C6D3F', sort=7, from_year=2018),
    dict(code='D8', name_vi='Quản trị điện tử',                         name_en='E-Governance',             short='QT điện tử',       color='#A63D57', sort=8, from_year=2018),
])
DIM_INDICATOR[['code','name_vi','color','from_year']]

**Nhận xét.** Tám trục nội dung, trong đó hai trục D7 và D8 chỉ có dữ liệu từ năm 2018.

## 3. Nhánh xử lý 2011-2017: tỉnh theo hàng

In [4]:
def parse_era1(path, year):
    xl = pd.ExcelFile(path)
    sheet = next(s for s in xl.sheet_names if 'tổng hợp' in s)
    df = pd.read_excel(path, sheet_name=sheet, header=0)
    df.columns = [str(c).strip() for c in df.columns]
    prov_col = df.columns[0]
    dim_cols = {c: f"D{re.match(r'^[1-6]', c.strip()).group()}"
                for c in df.columns if re.match(r'^\s*[1-6]\s*:', c)}
    out, n_prov = [], 0
    for _, r in df.iterrows():
        pid = to_pid(r[prov_col])
        if pid is None: continue
        n_prov += 1
        for c, dcode in dim_cols.items():
            v = pd.to_numeric(r[c], errors='coerce')
            if pd.notna(v): out.append((pid, year, dcode, float(v)))
    return out, n_prov, len(dim_cols)

recs, nprov, ndim = parse_era1(os.path.join(RAW, 'PAPI-2016-Dữ-liệu-1.xlsx'), 2016)
print(f'{nprov} tinh, {ndim} truc, {len(recs)} ban ghi')
pd.DataFrame(recs, columns=['province_id','year','code','score']).head(4)

63 tinh, 6 truc, 378 ban ghi


,province_id,year,code,score
0,15,2016,D1,5.337698
1,15,2016,D2,5.076181
2,15,2016,D3,4.261384
3,15,2016,D4,5.238945


**Nhận xét.** Nhánh thứ nhất đọc dữ liệu theo hàng-tỉnh và chuyển về dạng long. Giai đoạn này có sáu trục.

## 4. Nhánh xử lý 2018-2024: tỉnh theo cột

In [5]:
RE_DIM = re.compile(r'(?:Dimension|Chỉ số nội dung|nội dung)\s*([1-8])\s*:', re.IGNORECASE)

def parse_transposed(path, year, sheet):
    df = pd.read_excel(path, sheet_name=sheet, header=None)
    best_row, best_map = None, {}
    for r in range(min(6, len(df))):
        m = {c: to_pid(df.iloc[r, c]) for c in range(df.shape[1])}
        m = {c: p for c, p in m.items() if p is not None}
        if len(m) > len(best_map): best_row, best_map = r, m
    if not best_map: raise RuntimeError(f'{sheet}: khong tim duoc dong ten tinh')
    out, seen = [], set()
    for r in range(best_row + 1, df.shape[0]):
        lab = f"{df.iloc[r,0]} | {df.iloc[r,1] if df.shape[1]>1 else ''}"
        m = RE_DIM.search(lab)
        if m:
            dcode = f'D{m.group(1)}'
        else:
            l = lab.lower()
            is_total = ('unweighted papi score' in l) or \
                       ('papi tổng hợp' in l and 'không có trọng số' in l
                        and not any(k in l for k in ['điểm thấp','điểm cao','sai số','ci low','ci high','standard error']))
            dcode = 'TOTAL' if is_total else None
        if dcode is None or dcode in seen: continue
        seen.add(dcode)
        for c, pid in best_map.items():
            v = pd.to_numeric(df.iloc[r, c], errors='coerce')
            if pd.notna(v): out.append((pid, year, dcode, float(v)))
    return out, len(best_map), len([c for c in seen if c.startswith('D')]), ('TOTAL' in seen)

recs2, nprov2, ndim2, has_total = parse_transposed(
    os.path.join(RAW, '2023PAPI_ProvincialIndicators_BangChiTieuCapTinh..xlsx'), 2023, '2023_VIE_ENG')
print(f'{nprov2} tinh, {ndim2} truc, co tong={has_total}, {len(recs2)} ban ghi')
pd.DataFrame(recs2, columns=['province_id','year','code','score']).query("code=='D1'").head(3)

63 tinh, 8 truc, co tong=True, 549 ban ghi


,province_id,year,code,score
61,15,2023,D1,5.427534
62,1,2023,D1,5.283103
63,2,2023,D1,4.772978


**Nhận xét.** Nhánh thứ hai xử lý bố cục chuyển vị. Trục được nhận diện theo số thứ tự 1-8 thay vì theo nhãn văn bản, do nhãn thay đổi giữa tiếng Việt và tiếng Anh qua các năm. Chỉ giá trị Unweighted được giữ lại.

## 5. Gộp toàn bộ theo nguồn canonical mỗi năm

In [6]:
def year_sources():
    plan = [(y, 'era1', os.path.join(RAW, f'PAPI-{y}-Dữ-liệu-1.xlsx'), None) for y in range(2011, 2018)]
    plan += [
        (2018, 'trans', os.path.join(RAW, 'PAPI2018_ProvincialScores_ByIndicators_VIE.xlsx'), 'ProvincialIndicators_VIE'),
        (2019, 'trans', os.path.join(RAW, '2019_PAPI_Provincial_indicators2019_VIE_ENG.xlsx'), '2019PAPI_Indicators_VIE'),
        (2020, 'trans', os.path.join(RAW, '2020PAPI_ProvincialIndicators_BangChiTieuCapTinh-1.xlsx'), '2020_VIE_ENG'),
        (2021, 'trans', os.path.join(RAW, '1.2021PAPI_ProvincialIndicators_BangChiTieuCapTinh.xlsx'), '2021_VIE_ENG'),
        (2022, 'trans', os.path.join(RAW, '2022PAPI_ProvincialIndicators_BangChiTieuCapTinh.xlsx'), '2022_VIE_ENG'),
        (2023, 'trans', os.path.join(RAW, '2023PAPI_ProvincialIndicators_BangChiTieuCapTinh..xlsx'), '2023_VIE_ENG'),
        (2024, 'trans', os.path.join(RAW, '2024PAPI_ProvincialIndicators_BangChiTieuCapTinh_34_TinhThanh.xlsx'), '2024_PAPI_VIE_ENG'),
    ]
    return plan

records = []
summary = []
for year, era, path, sheet in year_sources():
    if era == 'era1':
        recs, nprov, ndim = parse_era1(path, year); has_total = False
    else:
        recs, nprov, ndim, has_total = parse_transposed(path, year, sheet)
    records.extend(recs)
    summary.append((year, nprov, ndim, has_total, len(recs)))
fact_raw = pd.DataFrame(records, columns=['province_id','year','code','score'])
pd.DataFrame(summary, columns=['nam','so_tinh','so_truc','co_tong','so_dong'])

,nam,so_tinh,so_truc,co_tong,so_dong
0,2011,63,6,False,378
1,2012,63,6,False,378
2,2013,63,6,False,378
3,2014,63,6,False,366
4,2015,63,6,False,378
5,2016,63,6,False,378
6,2017,63,6,False,378
7,2018,63,8,False,500
8,2019,63,8,True,567
9,2020,63,8,True,567


**Nhận xét.** Mỗi năm được lấy từ một nguồn chuẩn. Các tệp gốc theo năm được dùng thay cho tệp tổng hợp năm 2024, do tệp này tổ chức lại theo 34 tỉnh và làm rỗng dữ liệu một số tỉnh ở các sheet năm cũ.

## 6. Làm sạch: điểm 0 coi là thiếu, tách dòng tổng

In [7]:
zeros = fact_raw[fact_raw.score == 0]
print('So o diem 0 (coi la thieu):', len(zeros))
fact = fact_raw[fact_raw.score != 0].copy()

official_total = fact[fact.code == 'TOTAL'].rename(columns={'score':'total_official'}).drop(columns='code')
fact_dim = fact[fact.code != 'TOTAL'].drop_duplicates(['province_id','year','code']).copy()
fact_dim['province_id'] = fact_dim['province_id'].astype('int16')
fact_dim['year'] = fact_dim['year'].astype('int16')
fact_dim['code'] = fact_dim['code'].astype('category')
fact_dim['score'] = fact_dim['score'].astype('float32')
fact_dim = fact_dim.sort_values(['year','province_id','code']).reset_index(drop=True)
print('fact_dim:', fact_dim.shape); fact_dim.head(4)

So o diem 0 (coi la thieu): 28
fact_dim: (6094, 4)


,province_id,year,code,score
0,1,2011,D1,4.876205
1,1,2011,D2,4.829181
2,1,2011,D3,5.007073
3,1,2011,D4,5.220708


**Nhận xét.** Giá trị 0 không hợp lệ trên thang 1-10 nên được xem là thiếu. Dòng tổng được tách riêng để đối chiếu ở bước kiểm tra chất lượng.

## 7. Tổng hợp: bảng wide đầy đủ 63 tỉnh và 14 năm, và bảng cả nước

In [8]:
DIM_ORDER = [f'D{i}' for i in range(1, 9)]
years = list(range(2011, 2025))
full = pd.MultiIndex.from_product([DIM_PROVINCE.province_id, years], names=['province_id','year'])
wide = (fact_dim.pivot_table(index=['province_id','year'], columns='code', values='score', observed=False)
        .reindex(full).reset_index())
for c in DIM_ORDER:
    if c not in wide.columns: wide[c] = pd.NA
wide['n_dims'] = wide[DIM_ORDER].notna().sum(axis=1)
wide['expected_dims'] = wide.year.apply(lambda y: 6 if y <= 2017 else 8)
wide['total_papi'] = wide[DIM_ORDER].sum(axis=1, min_count=1)
wide.loc[wide.n_dims < wide.expected_dims, 'total_papi'] = pd.NA
# total_papi_6dim: tong D1-D6, so sanh lien mach 2011-2024 (ne moc 6->8 truc nam 2018)
DIM6 = [f'D{i}' for i in range(1, 7)]
wide['total_papi_6dim'] = wide[DIM6].sum(axis=1, min_count=6)
wide = wide.merge(official_total, on=['province_id','year'], how='left')
wide['rank_year'] = wide.groupby('year')['total_papi'].rank(ascending=False, method='min').astype('Int64')
wide['tier'] = wide.groupby('year')['total_papi'].transform(
    lambda s: pd.qcut(s, 4, labels=['Thấp nhất','TB thấp','TB cao','Cao nhất']))
wide = wide.merge(DIM_PROVINCE[['province_id','province_vi','region','region_id']], on='province_id', how='left')
wide = wide[['province_id','province_vi','region','region_id','year'] + DIM_ORDER +
            ['total_papi','total_papi_6dim','total_official','n_dims','rank_year','tier']].sort_values(['year','province_id'])

nat = (fact_dim.groupby(['year','code'], observed=False)['score']
       .agg(mean_score='mean', min_score='min', max_score='max', std_score='std').reset_index())

gaps = wide[wide.total_papi.isna()][['province_vi','year','n_dims']]
print('So tinh-nam thieu du lieu that tai nguon:', len(gaps))
gaps

So tinh-nam thieu du lieu that tai nguon: 13


,province_vi,year,n_dims
115,Bắc Giang,2014,0
773,Đồng Tháp,2014,0
245,Quảng Ninh,2018,6
777,Đồng Tháp,2018,6
122,Bắc Giang,2021,0
234,Bắc Ninh,2021,0
248,Quảng Ninh,2021,0
123,Bắc Giang,2022,4
235,Bắc Ninh,2022,4
250,Quảng Ninh,2023,0


**Nhận xét.** Bảng wide là panel đầy đủ 63 nhân 14 bằng 882 dòng. Tổng điểm để trống khi tỉnh không đủ trục, nhằm không báo cáo tổng sai. Có 13 tỉnh-năm thiếu dữ liệu sẵn có trong nguồn; các trường hợp này được giữ ở dạng thiếu, không thay thế bằng giá trị suy đoán.

## 8. Kiểm tra chất lượng

In [9]:
checks = []
def check(name, ok, detail=''):
    checks.append(ok); print(('PASS' if ok else 'FAIL'), '|', name, '|', detail)

check('Tong so dong long >= 2000', len(fact_dim) >= 2000, f'= {len(fact_dim)}')
ppy = fact_dim.groupby('year')['province_id'].nunique()
check('Moi nam >= 60 tinh', ppy.min() >= 60, f'min={ppy.min()}, max={ppy.max()}')
check('Panel wide = 882 dong', len(wide) == 882, f'= {len(wide)}')
check('Diem truc trong (0,10]', (fact_dim.score.gt(0) & fact_dim.score.le(10)).all(),
      f'min={fact_dim.score.min():.2f}, max={fact_dim.score.max():.2f}')
check('Tong PAPI trong [10,80]', wide.total_papi.dropna().between(10,80).all(),
      f'min={wide.total_papi.min():.1f}, max={wide.total_papi.max():.1f}')
cmp = wide.dropna(subset=['total_official']).query('n_dims == 8')
diff = (cmp.total_papi - cmp.total_official).abs()
check('Tong tu tinh = tong official', (diff < 0.01).all(), f'lech max = {diff.max():.4f}')
print('\nTat ca dat:', all(checks))

PASS | Tong so dong long >= 2000 | = 6094
PASS | Moi nam >= 60 tinh | min=60, max=63
PASS | Panel wide = 882 dong | = 882
PASS | Diem truc trong (0,10] | min=1.93, max=8.46
PASS | Tong PAPI trong [10,80] | min=31.7, max=48.8
PASS | Tong tu tinh = tong official | lech max = 0.0000

Tat ca dat: True


**Nhận xét.** Phép đối chiếu quan trọng nhất cho thấy tổng điểm tự tính trùng khớp với tổng do nguồn công bố, với sai lệch không đáng kể, xác nhận quá trình parser chính xác.

## 9. Đối chiếu với bản trong src và xuất kết quả

In [10]:
# Doi chieu ket qua notebook voi ban cai dat trong src/ (dung cho ung dung dashboard)
import sys, io, contextlib
sys.path.insert(0, os.path.join(ROOT, 'src'))
import build_dataset as bd, papi_lib as L
with contextlib.redirect_stdout(io.StringIO()):
    fact_src, _ = bd.clean(bd.parse_all())
key = ['year','province_id','code']
a = fact_dim.astype({'code':'object'}).sort_values(key).reset_index(drop=True)
b = fact_src.astype({'code':'object'}).sort_values(key).reset_index(drop=True)
same = (len(a) == len(b)
        and (a[['province_id','year']].values == b[['province_id','year']].values).all()
        and (a['code'].values == b['code'].values).all()
        and (a['score'].round(5).values == b['score'].round(5).values).all())
print('Doi chieu fact_dim voi src/build_dataset:', 'KHOP' if same else 'KHAC')
assert same, 'Notebook va src cho fact_dim khac nhau, can dong bo lai'

# Doi chieu them hai bang tra cuu (bat drift ve cot/gia tri)
ci, cp = list(DIM_INDICATOR.columns), list(DIM_PROVINCE.columns)
assert DIM_INDICATOR[ci].values.tolist() == L.DIM_INDICATOR[ci].values.tolist(), 'DIM_INDICATOR lech src'
assert DIM_PROVINCE[cp].values.tolist() == L.DIM_PROVINCE[cp].values.tolist(), 'DIM_PROVINCE lech src'
print('Doi chieu dim_indicator, dim_province voi src: KHOP')

# Xuat file
fact_dim.to_parquet(os.path.join(PROC, 'fact_papi_long.parquet'), index=False)
fact_dim.to_csv(os.path.join(PROC, 'fact_papi_long.csv'), index=False)
wide.to_parquet(os.path.join(PROC, 'agg_province_year.parquet'), index=False)
wide.to_csv(os.path.join(PROC, 'agg_province_year.csv'), index=False)
nat.to_parquet(os.path.join(PROC, 'agg_national_year.parquet'), index=False)
DIM_PROVINCE.to_csv(os.path.join(PROC, 'dim_province.csv'), index=False)
DIM_INDICATOR.to_csv(os.path.join(PROC, 'dim_indicator.csv'), index=False)
print('Da xuat data/processed/')

Doi chieu fact_dim voi src/build_dataset: KHOP
Doi chieu dim_indicator, dim_province voi src: KHOP
Da xuat data/processed/


**Nhận xét.** Kết quả của notebook trùng khớp hoàn toàn với bản cài đặt trong `src/`. Nhờ vậy, có thể chạy notebook này hoặc chạy `python src/build_dataset.py`, cả hai cho cùng một bộ dữ liệu. Thư mục `src/` được giữ làm thư viện cho ứng dụng dashboard và để đối chiếu; notebook là bản trình bày và kiểm chứng đầy đủ logic. Bước tiếp theo là `eda.ipynb`.